# snmCT Mapping Summary

This notebook provides a quick overview of key mapping metrics from the hisat3n pipeline. You can customize it as needed.

## Parameters

## Prepare

In [ ]:
# parameters
output_dir = ''
plate_col = 'Plate'
color_quantile = (0.05, 0.95)

### Load

In [ ]:
import pathlib
import pandas as pd
from cemba_data.utilities import get_configuration

output_dir = pathlib.Path(output_dir)
mapping_summary = pd.read_csv(output_dir / 'stats/MappingSummary.csv.gz', index_col=0)
config = get_configuration(output_dir / 'mapping_config.ini')

In [ ]:
mapping_summary.columns

### Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from cemba_data.summary import cutoff_vs_cell_remain, plot_on_plate


def distplot_and_plate_view(data, hue, color_quantile=color_quantile, config=config):
    fig1, (vmin, vmax) = cutoff_vs_cell_remain(data=data[hue].dropna(), 
                                               bins=50, kde=False,
                                               xlim_quantile=color_quantile)

    fig2, plate_names, plate_datas = plot_on_plate(
        data=data,
        hue=hue,
        groupby=plate_col,
        vmin=vmin,
        vmax=vmax,
        aggregation_func=lambda i: i.mean())
    
    fig3, ax = plt.subplots(figsize=(data[plate_col].unique().size * 2, 4))
    plate_hue_name = 'Plate'
    sns.boxenplot(data=data, x=plate_col, y=hue, palette='hls', 
                  ax=ax, hue=plate_hue_name)
    ax.set_ylim(vmin, vmax)
    ax.xaxis.set_tick_params(rotation=90)
    ax.legend(bbox_to_anchor=(1.1, 1), title=plate_hue_name)
    sns.despine(ax=ax)
    return

In [ ]:
# plot defaults
sns.set_context(context='notebook')
plt.rc('figure', dpi=150)

## Summary

In [ ]:
# conventional basic check, change as you need
mccc_cutoff = 0.03
high_mccc = mapping_summary['mCCCFrac'] > mccc_cutoff

miseq_guess = mapping_summary['UniqueAlignFinalDNAReads'].mean() < 50000
reads_cutoff = 100 if miseq_guess else 500000
low_reads = mapping_summary['UniqueAlignFinalDNAReads'] < reads_cutoff

success = ~high_mccc & ~low_reads
n_cell = mapping_summary.shape[0]
n_plate = mapping_summary['Plate'].unique().size
total_wells = n_plate * 384

In [ ]:
print(f"""
This library seems to be a {'MiSeq' if miseq_guess else 'NovaSeq'} library.

Cells
    {n_plate}\t plates
    {total_wells}\t wells (total cell number in theory)

    {n_cell} ({n_cell / total_wells * 100:.1f}%)\t cells having mapping metric
    {success.sum()} ({success.sum() / total_wells * 100:.1f}%)\t cells passed basic QC (mCCC and # of final reads)
    {high_mccc.sum()} ({high_mccc.sum() / total_wells * 100:.1f}%)\tcells having high mCCC frac (> {mccc_cutoff})
    {low_reads.sum()} ({low_reads.sum() / total_wells * 100:.1f}%)\tcells having low number of final DNA reads (< {reads_cutoff}).

Reads
    {mapping_summary['InputReadPairs'].sum()*2:.0f}\tTotal Input Reads (R1 & R2)
    {mapping_summary['InputReadPairs'].mean()*2:.0f}\tAverage Input Reads per cell (R1 & R2)
    
    {mapping_summary['UniqueAlignFinalDNAReads'].sum():.0f}\tTotal Final DNA Reads (unique align)
    {mapping_summary['UniqueAlignFinalDNAReads'].mean():.0f}\tAverage Final DNA Reads per cell (unique align)
    {mapping_summary['FinalRNAReads'].sum():.0f}\tTotal Final RNA Reads
    {mapping_summary['FinalRNAReads'].mean():.0f}\tAverage Final RNA Reads per cell

Mapping Rate
    {mapping_summary['DNAReadsUniqueMappingRate'].mean():.1f}%\tDNA Unique Mapping Rate
    {mapping_summary['RNAReadsUniqueMappingRate'].mean():.1f}%\tRNA Unique Mapping Rate

PCR Duplication
    {mapping_summary['DNAUniqueAlignPCRDuplicationRate'].mean():.1f}%\tDNA Unique Align PCR Duplication Rate
""")

## Reads Yield

### DNA Yield

In [ ]:
distplot_and_plate_view(mapping_summary, hue='DNAReadsYield')

### RNA Yield

In [ ]:
distplot_and_plate_view(mapping_summary, hue='RNAReadsYield')

### RNA / (DNA + RNA)

In [ ]:
distplot_and_plate_view(mapping_summary, hue='RNA/(DNA+RNA)')

## mC Fraction

### mCCC

In [ ]:
distplot_and_plate_view(mapping_summary, hue='mCCCFrac')

### mCH

In [ ]:
distplot_and_plate_view(mapping_summary, hue='mCHFrac')

### mCG

In [ ]:
distplot_and_plate_view(mapping_summary, hue='mCGFrac')

## FASTQ Metric

### InputReadPairs

In [ ]:
distplot_and_plate_view(mapping_summary, hue='InputReadPairs')

### TrimmedReadPairs

In [ ]:
distplot_and_plate_view(mapping_summary, hue='TrimmedReadPairs')

## Mapping Rate

### DNA Unique Mapping Rate

In [ ]:
distplot_and_plate_view(mapping_summary, hue='DNAReadsUniqueMappingRate')

### RNA Unique Mapping Rate

In [ ]:
distplot_and_plate_view(mapping_summary, hue='RNAReadsUniqueMappingRate')

## PCR Duplication Rate

### DNA PCR Duplication Rate

In [ ]:
distplot_and_plate_view(mapping_summary, hue='DNAUniqueAlignPCRDuplicationRate')

## Final Reads

### DNA (mC) Reads

In [ ]:
distplot_and_plate_view(mapping_summary, hue='UniqueAlignFinalDNAReads')

### RNA Reads

In [ ]:
distplot_and_plate_view(mapping_summary, hue='FinalRNAReads')

### RNA Assigned Reads Rate

In [ ]:
distplot_and_plate_view(mapping_summary, hue='AssignedRNAReadsRate')

## Lambda phage DNA non-conversion and over-conversion rate estimates

### Lambda CH - non-conversion estimate

Lambda DNA is not methylated on CH, so mCH is a measure of non-conversion

In [ ]:
distplot_and_plate_view(mapping_summary, hue='LambdamCHFrac')

### Lambda CG - over-conversion estimate

Lambda DNA was methylated with enzymatic CG methyl-transferase. So mCG below 100% is an indication of overconversion or possibly insufficient enzymatic CG methylation 

In [ ]:
distplot_and_plate_view(mapping_summary, hue='LambdamCGFrac')

## Mapping config

In [ ]:
config